In [ ]:
!pip install tensorflow pandas numpy scikit-learn matplotlib

import pandas as pd
import numpy as np
import os
import cv2
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50

In [ ]:
# Load CSV file
data = pd.read_csv("/content/sample_data/california_housing_train.csv")

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
data.shape

In [ ]:
data.columns

In [ ]:
data.isnull().sum()

**STEP 2 — Normalize & Split**

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

data = pd.read_csv("/content/sample_data/california_housing_train.csv")

X_tabular = data[['longitude','latitude','housing_median_age',
                  'total_rooms','total_bedrooms','population',
                  'households','median_income']].values

y = data['median_house_value'].values

scaler = StandardScaler()
X_tabular = scaler.fit_transform(X_tabular)

X_tab_train, X_tab_test, y_train, y_test = train_test_split(
    X_tabular, y, test_size=0.2, random_state=42
)

**✅ STEP 3 — Create tf.data Dataset (Memory Safe)**

In [ ]:
import tensorflow as tf

IMAGE_SIZE = 64   # smaller = safer
BATCH_SIZE = 8

def create_dataset(tabular, labels):
    dataset = tf.data.Dataset.from_tensor_slices((tabular, labels))

    def add_dummy_image(tab, label):
        image = tf.zeros((IMAGE_SIZE, IMAGE_SIZE, 3))
        return (image, tab), label

    dataset = dataset.map(add_dummy_image)
    dataset = dataset.batch(BATCH_SIZE)
    return dataset

train_dataset = create_dataset(X_tab_train, y_train)
test_dataset = create_dataset(X_tab_test, y_test)

**✅ STEP 4 — Build Image CNN (Lightweight)**

In [ ]:
from tensorflow.keras import layers, models

# Image branch
image_input = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))

x = layers.Conv2D(16, (3,3), activation='relu')(image_input)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(32, (3,3), activation='relu')(x)
x = layers.MaxPooling2D()(x)

x = layers.Flatten()(x)
image_output = layers.Dense(32, activation='relu')(x)

**✅ STEP 5 — Tabular Branch**

In [ ]:
tabular_input = layers.Input(shape=(X_tab_train.shape[1],))

t = layers.Dense(64, activation='relu')(tabular_input)
t = layers.Dense(32, activation='relu')(t)

**✅ STEP 6 — Combine Both Modalities**

In [ ]:
combined = layers.concatenate([image_output, t])

z = layers.Dense(32, activation='relu')(combined)
z = layers.Dense(1)(z)   # Regression output

model = models.Model(inputs=[image_input, tabular_input], outputs=z)

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

model.summary()

**✅ STEP 7 — Train Model**

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=10
)

**✅ STEP 8 — Evaluate (MAE & RMSE)**

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

predictions = model.predict(test_dataset)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print("MAE:", mae)
print("RMSE:", rmse)